# 03. 전처리: Model Drift 제거 및 SSTA 계산

---
### 1. CESM-HR 전처리
CESM-HR : 방사 강제력(radiative forcing)에 의한 추세는 없지만 **model drift** (수백 년에 걸친 완만한 drift)가 존재.
```
X_raw(i, t)   → [선형 추세 제거]  → X_detr(i, t) = X_raw(i,t) − (a_i + b_i·t)
X_detr(i,m,y) → [월별 기후장 제거] → X'(i,m,y)    = X_detr(i,m,y) − X̄_detr(i,m)
```
- `i` : 격자점, `t` : 전체 시간축(월 인덱스), `m` : 달(1–12), `y` : 연도
- `X̄_detr(i,m)` : 선형 추세 제거 후 월별 기후장 (full-record 평균)

### 2. OISST 전처리
OISST는 아노말리를 계산하고 격자점별 선형 추세를 제거 (CESM-HR이 radiatively forced trend가 없으므로 관측에서도 제거)

```
O'(i,t) = O(i,t) − Ō(i,m) − (a_i + b_i·t)
```
순서가 결과에 미치는 영향을 보기 위해 **두 버전**을 모두 만든다:
- **v1**: 기후장 제거 먼저 → 선형 추세 제거
- **v2**: 선형 추세 제거 먼저 → 기후장 제거

### 3. 격자 맞추기
Analog selection 직전에 CESM-HR SSTA(curvilinear POP grid)를
OISST의 0.25°×0.25° 규칙 격자로 regrid한다.

---

**입력 파일** (`data/processed/`):
- `cesm_hr_sst_domain{a,b,c}.nc`
- `oisst_monthly_nwpacific_domain{a,b,c}_1993_2024.nc`

**출력 파일** (`data/processed/`):
- `cesm_hr_ssta_domain{a,b,c}.nc` — CESM-HR SSTA (detrend + climatology 제거)
- `oisst_ssta_v1_domain{a,b,c}.nc` — OISST SSTA v1 (anomaly → detrend)
- `oisst_ssta_v2_domain{a,b,c}.nc` — OISST SSTA v2 (detrend → anomaly)
- `cesm_hr_ssta_regridded_domain{a,b,c}.nc` — CESM-HR SSTA regridded to OISST grid

## 0. 임포트 및 경로 설정

In [2]:
import xarray as xr
import numpy as np
import nc_time_axis
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path
from scipy.interpolate import LinearNDInterpolator  # regrid

PROC_DIR = Path("../data/processed")
FIG_DIR  = Path("../figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

# OISST와 CESM-HR에 대해 동일한 이름(a/b/c)으로 짝을 맞춤
DOMAINS = ["local_wnp", "trop_wnp", "north_pacific"]

CLIM_START = "1993"
CLIM_END   = "2024"

## 1. 데이터 로드

전처리된 월평균 SST 파일을 읽어 딕셔너리로 저장한다.

In [ ]:
# ── CESM-HR 로드 ──────────────────────────────────────────────────────
# 변수: SST (degC), 좌표: TLAT/TLONG (2D 곡선격자), time (cftime 객체)
# 시간: pre-industrial control year 250–349 → 1,200 개월
cesm = {}
for d in DOMAINS:
    ds = xr.open_dataset(PROC_DIR / f"cesm_hr_sst_{d}.nc")
    cesm[d] = ds["SST"]
    print(f"CESM-HR domain {d}: {dict(ds['SST'].sizes)}")
    print(f"  TLAT  range: {float(ds.TLAT.min()):.2f} ~ {float(ds.TLAT.max()):.2f}")
    print(f"  TLONG range: {float(ds.TLONG.min()):.2f} ~ {float(ds.TLONG.max()):.2f}")
    ds.close()

print()

# ── OISST 로드 ────────────────────────────────────────────────────────
# 변수: sst (Celsius), 좌표: lat/lon (1D 규칙 격자 0.25°), time (datetime64)
# 시간: 1993-01 ~ 2024-12 → 384 개월
oisst = {}
for d in DOMAINS:
    ds = xr.open_dataset(PROC_DIR / f"oisst_monthly_nwpacific_{d}_1993_2024.nc")
    oisst[d] = ds["sst"]
    print(f"OISST domain {d}: {dict(ds['sst'].sizes)}")
    print(f"  lat range: {float(ds.lat.min()):.3f} ~ {float(ds.lat.max()):.3f}")
    print(f"  lon range: {float(ds.lon.min()):.3f} ~ {float(ds.lon.max()):.3f}")
    ds.close()

print("\n데이터 로드 완료")

## 2. CESM-HR 전처리

### 배경: Model Drift란?
CESM-HR은 수백 년짜리 pre-industrial control run이다. 실제 대기 조성(CO₂ 등)이 고정되어 있어
방사 강제력에 의한 장기 추세는 없지만, **모델이 초기 조건에서 평형 상태로 서서히 이동**하는
완만한 drift가 생긴다. 이를 제거하지 않으면 나중에 OISST 아노말리와 비교할 때
두 자료 사이에 인위적인 바이어스가 생긴다.

### 처리 순서
1. **선형 추세 제거** (full-record linear detrend): 격자점별로 1200개월 시계열에 직선을 맞추고 제거
2. **월별 기후장 제거** (monthly climatology): 선형 추세 제거 후 각 달의 평균을 제거하여 계절 신호 제거

### 2.1 선형 추세 제거 함수 정의

각 격자점 `i`에서 다음을 계산한다:
```
X_detr(i, t) = X_raw(i, t) − (a_i + b_i · t)
```
여기서 `a_i, b_i`는 OLS(최소자승법)으로 구한 절편과 기울기이다.

**구현 방법**: `xr.apply_ufunc(vectorize=True)` 로 각 격자점의 1D 시계열에 `np.polyfit`을 적용한다.
- `vectorize=True` → xarray가 공간 격자별로 반복 호출 (NaN 안전)
- `input_core_dims=[[dim]]` → 시간축을 'core' 차원으로 지정 (나머지 차원은 자동 루프)
- CESM-HR 육지 격자(NaN)는 그대로 NaN 반환

In [ ]:
def linear_detrend(da, dim="time"):
    """
    DataArray의 각 격자점에서 선형 추세를 제거한다.

    Parameters
    ----------
    da  : xr.DataArray  — 입력 데이터 (시간 차원을 포함해야 함)
    dim : str           — 시간 차원 이름 (기본값: "time")

    Returns
    -------
    xr.DataArray — 선형 추세가 제거된 배열 (원본과 shape·dim 순서 동일)

    주의
    ----
    xr.apply_ufunc(output_core_dims=[[dim]])은 core 차원(time)을
    결과 배열의 **마지막 축**으로 이동시킨다.
    함수 끝에서 .transpose(*da.dims)로 원래 순서를 복원한다.
    """
    n = da.sizes[dim]

    # 시간 인덱스: 0, 1, ..., N-1
    # 중심화(centering)하면 OLS에서 기울기와 절편이 독립이 되어 수치 안정성↑
    t = np.arange(n, dtype=np.float64)
    t_c = t - t.mean()

    def _detrend_1d(x):
        """1D 시계열 x에서 선형 추세 제거 (NaN = 육지 격자 → NaN 반환)"""
        mask = np.isfinite(x)
        if mask.sum() < 2:
            return np.full_like(x, np.nan)
        p     = np.polyfit(t_c[mask], x[mask], 1)   # [기울기 b_i, 절편 a_i]
        trend = p[0] * t_c + p[1]                    # 선형 추세선
        return x - trend

    result = xr.apply_ufunc(
        _detrend_1d,
        da,
        input_core_dims=[[dim]],    # dim(시간)을 core로 → 함수에 1D로 전달
        output_core_dims=[[dim]],   # 출력도 같은 시간축 유지
        vectorize=True,             # 공간 격자별로 _detrend_1d 반복 호출
        dask="parallelized",
        output_dtypes=[da.dtype],
    )
    # ── 핵심 수정 ──────────────────────────────────────────────────────
    # apply_ufunc은 output_core_dims를 결과 배열의 마지막 축으로 이동시킴
    # 예: 입력 (time, nlat, nlon) → 출력 (nlat, nlon, time)
    # .transpose(*da.dims)로 원본 차원 순서를 복원
    return result.transpose(*da.dims)


print("linear_detrend 함수 정의 완료")
print("(apply_ufunc 후 .transpose(*da.dims)로 차원 순서 복원)")


### 2.2 CESM-HR 선형 추세 제거

세 도메인(a, b, c) 각각에 대해 선형 추세를 제거한다.

> ⚠️ **계산 시간 주의**: domain a (517×750 격자)는 약 2–5분 소요될 수 있다.

In [ ]:
cesm_detr = {}  # 선형 추세 제거 결과 저장

for d in DOMAINS:
    print(f"domain {d} 선형 추세 제거 중...", end=" ", flush=True)

    # CESM-HR은 cftime 객체이므로 시간축은 자동으로 처리됨
    # (t는 0,1,...,N-1 인덱스를 사용하기 때문에 cftime 여부와 무관)
    cesm_detr[d] = linear_detrend(cesm[d], dim="time")
    cesm_detr[d] = cesm_detr[d].compute()  # Dask lazy 연산 실행

    print("완료")

print("\n전체 선형 추세 제거 완료")

In [ ]:
# ── 선형 추세 제거 검증: 임의 격자점의 원본 vs 추세 제거 후 시계열 비교 ──
# domain a에서 해양 격자(NaN이 없는 격자)를 하나 선택해서 시각화

d_check = "north_pacific"  # 작은 domain c로 빠르게 확인
raw  = cesm[d_check]
detr = cesm_detr[d_check]

# 해양 격자 찾기: 모든 시간에서 NaN이 아닌 격자 (바다)
ocean_mask = np.isfinite(raw.isel(time=0).values)
rows, cols  = np.where(ocean_mask)
# 중간 격자 선택
ri, ci = rows[len(rows)//2], cols[len(cols)//2]

raw_1d  = raw.isel(nlat=ri, nlon=ci).values
detr_1d = detr.isel(nlat=ri, nlon=ci).values
t_idx   = np.arange(len(raw_1d))

# 원본의 선형 추세선 계산 (시각화용)
p_orig = np.polyfit(t_idx - t_idx.mean(), raw_1d, 1)
trend_line = np.polyval(p_orig, t_idx - t_idx.mean())

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

# 위: 원본 + 추세선
ax = axes[0]
ax.plot(t_idx, raw_1d,   color="steelblue", lw=0.8, label="raw SST")
ax.plot(t_idx, trend_line, color="red",   lw=2,   label=f"linear trend (slope: {p_orig[0]*120:.4f} °C/10yr)")
ax.set_ylabel("SST (°C)")
ax.set_title(f"CESM-HR domain {d_check} — Ocean grid (nlat={ri}, nlon={ci})")
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# 아래: 추세 제거 후
ax = axes[1]
ax.plot(t_idx, detr_1d,  color="darkorange", lw=0.8, label="추세 제거 후")
ax.axhline(0, color="black", lw=0.8, ls="--")
ax.set_xlabel("Time index (M)")
ax.set_ylabel("SST anomaly (°C)")
ax.set_title("Declining linear trend -> SST")
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(FIG_DIR / f"cesm_detrend_check_domain{d_check}.png", dpi=150)
plt.show()
print(f"\nbf declination of linear trend std: {np.nanstd(raw_1d):.4f} °C")
print(f"af declination of linear trend std: {np.nanstd(detr_1d):.4f} °C")

### 2.3 월별 기후장(Monthly Climatology) 제거

선형 추세가 제거된 시계열에서 계절 사이클을 제거한다:
```
X'(i, m, y) = X_detr(i, m, y) − X̄_detr(i, m)
```
- `X̄_detr(i, m)` : domain별, 격자점별 **월별 long-term mean** (1200개월 full-record 기준)
- `m` : 달(1=1월, 2=2월, ..., 12=12월)

**구현**: `da.groupby('time.month').mean('time')` 으로 기후장 계산 후,
`da.groupby('time.month') - climatology` 로 제거한다.

In [ ]:
cesm_ssta = {}  # 최종 CESM-HR SSTA 저장

for d in DOMAINS:
    da = cesm_detr[d]  # 선형 추세 제거된 SST

    # ── Step 1: 월별 기후장 계산 ────────────────────────────────────────
    # groupby('time.month'): 1~12월로 그룹화
    # .mean('time'): 각 달의 전체 시간 평균 → shape: (12, nlat, nlon)
    clim = da.groupby("time.month").mean("time")
    print(f"domain {d} 기후장 shape: {dict(clim.sizes)}")

    # ── Step 2: 기후장 제거 (월별로 대응) ───────────────────────────────
    # groupby를 사용하면 xarray가 자동으로 각 time step의 month에 맞는
    # 기후장을 빼준다 (broadcasting 자동 처리)
    ssta = da.groupby("time.month") - clim

    # 'month' 좌표가 추가되었을 수 있으므로 drop
    if "month" in ssta.coords:
        ssta = ssta.drop_vars("month")

    # 메타데이터 추가
    ssta.attrs.update({
        "long_name"   : "Sea Surface Temperature Anomaly (CESM-HR)",
        "units"       : "degC",
        "processing"  : "1) full-record linear detrend per grid cell, "
                        "2) monthly climatology removed (full-record mean)",
        "domain"      : d,
    })

    cesm_ssta[d] = ssta
    print(f"domain {d} SSTA: {dict(ssta.sizes)}")
    print(f"  mean={float(ssta.mean()):.4f}  std={float(ssta.std()):.4f}  "
          f"NaN%={float(ssta.isnull().mean())*100:.1f}%")
    print()

print("CESM-HR SSTA 계산 완료")

In [ ]:
# ── 검증: 기후장 제거가 제대로 됐는지 확인 ───────────────────────────────
# 목표: SSTA의 월별 mean이 0에 가까워야 함 (± 부동소수점 오차)

d_check = "north_pacific"
ssta = cesm_ssta[d_check]

# 공간 평균된 월별 시계열
ssta_spatial_mean = ssta.mean(dim=["nlat", "nlon"])  # (time,) — NaN은 무시

# 월별 평균 (SSTA의 월별 평균이 0인지 확인)
monthly_bias = ssta_spatial_mean.groupby("time.month").mean("time")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# 좌: 공간 평균 SSTA 시계열
ax = axes[0]
ssta_spatial_mean.plot(ax=ax, lw=0.8, color="steelblue")
ax.axhline(0, color="black", lw=0.8, ls="--")
ax.set_title(f"CESM-HR domain {d_check} — Spatial Mean SSTA")
ax.set_ylabel("SSTA (°C)")
ax.grid(alpha=0.3)

# 우: 월별 바이어스 (거의 0이어야 정상)
ax = axes[1]
ax.bar(monthly_bias.month.values, monthly_bias.values, color="steelblue", edgecolor="white")
ax.axhline(0, color="red", lw=1)
ax.set_xticks(range(1, 13))
ax.set_xticklabels(["Jan","Feb","Mar","Apr","May","Jun",
                    "Jul","Aug","Sep","Oct","Nov","Dec"])
ax.set_title("Monthly SSTA Bias (should be ≈0)")
ax.set_ylabel("Monthly SSTA Mean (°C)")
ax.grid(alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig(FIG_DIR / f"cesm_ssta_bias_check_{d_check}.png", dpi=150)
plt.show()

print(f"월별 바이어스 최대값: {float(abs(monthly_bias).max()):.6f} °C  (float32 반올림 오차 수준이어야 정상)")

In [ ]:
# ── CESM-HR SSTA 저장 ─────────────────────────────────────────────────
print("=== CESM-HR SSTA 저장 ===")
for d in DOMAINS:
    out_path = PROC_DIR / f"cesm_hr_ssta_{d}.nc"
    if out_path.exists():
        out_path.unlink()  # 덮어쓰기 방지

    cesm_ssta[d].to_dataset(name="SSTA").to_netcdf(
        out_path,
        encoding={"SSTA": {"zlib": True, "complevel": 4, "dtype": "float32"}},
    )
    print(f"저장 완료: {out_path.name}  {dict(cesm_ssta[d].sizes)}")

print("\n저장 파일 확인:")
for d in DOMAINS:
    ds_chk = xr.open_dataset(PROC_DIR / f"cesm_hr_ssta_{d}.nc")
    print(f"  domain {d}: {dict(ds_chk['SSTA'].sizes)}  "
          f"NaN {float(ds_chk['SSTA'].isnull().mean())*100:.1f}%")
    ds_chk.close()

## 3. OISST 전처리

OISST는 관측 자료로, analog selection에서 **초기 관측 상태**를 나타낸다.
CESM-HR이 pre-industrial control (radiatively forced trend 없음)이므로,
OISST에서도 1994–2024 기간의 온난화 추세를 제거해야 두 자료를 비교할 수 있다.

### 기후장 기준: 1994–2019
- 관측 기간 전체 평균 대신 고정 기준 기간을 사용
- 1994: OISST 안정적인 데이터 시작 연도 (AVHRR 위성 자료 안정화)
- 2019: 기후장 계산 종료 연도 (최근 수년을 제외해 기후 변화 영향 최소화)

### 두 버전 비교
수식상 두 연산(기후장 제거, 선형 추세 제거)은 **가환적(commutative)이지 않다**.
선형 추세가 기후장에 흡수되는 정도가 다르기 때문이다.

- **v1** (`anomaly → detrend`): 먼저 1994–2019 climatology 제거 → 아노말리에서 선형 추세 제거
- **v2** (`detrend → anomaly`): 먼저 선형 추세 제거 → 추세 제거된 1994–2019 climatology 제거

### 3.1 OISST Version 1: 기후장 먼저 제거 → 선형 추세 제거

```
O_anom(i,t) = O(i,t)      − Ō₁₉₉₄₋₂₀₁₉(i,m)       ← 계절 기후장 제거
O'_v1(i,t)  = O_anom(i,t) − (a_i + b_i·t)            ← 아노말리의 선형 추세 제거
```
이 방법은 기후장이 전체 트렌드(온난화)를 일부 흡수한 상태에서 아노말리를 구하고,
그 아노말리에 남아 있는 선형 추세를 별도로 제거한다.

In [ ]:
oisst_ssta_v1 = {}  # Version 1 결과 저장

for d in DOMAINS:
    da = oisst[d]  # 원본 OISST 월평균 SST

    # ── Step 1: 1994–2019 월별 기후장 계산 ─────────────────────────────
    # 1994–2019 기간만 선택해서 월별 평균 계산
    # → shape: (12, lat, lon)
    clim_period = da.sel(time=slice(CLIM_START, CLIM_END))
    clim_v1     = clim_period.groupby("time.month").mean("time")

    # ── Step 2: 기후장 제거 → 아노말리 ────────────────────────────────
    # 1993–2024 전체 기간에 대해 해당 월의 기후장을 제거
    # (1993년도 데이터도 1994–2019 기후장을 기준으로 아노말리 계산)
    anom_v1 = da.groupby("time.month") - clim_v1
    if "month" in anom_v1.coords:
        anom_v1 = anom_v1.drop_vars("month")

    # ── Step 3: 아노말리에서 선형 추세 제거 ────────────────────────────
    # 아노말리 시계열에 여전히 온난화 추세가 남아 있을 수 있음
    # → CESM-HR(pre-industrial)과 비교하기 위해 제거
    ssta_v1 = linear_detrend(anom_v1, dim="time")
    ssta_v1 = ssta_v1.compute()

    ssta_v1.attrs.update({
        "long_name"  : "OISST SSTA v1 (anomaly → detrend)",
        "units"      : "degC",
        "processing" : f"1) monthly climatology removed ({CLIM_START}–{CLIM_END}), "
                       "2) linear detrend per grid cell",
    })

    oisst_ssta_v1[d] = ssta_v1
    print(f"OISST v1 domain {d}: {dict(ssta_v1.sizes)}  "
          f"mean={float(ssta_v1.mean()):.4f}  std={float(ssta_v1.std()):.4f}")

print("\nOISST v1 계산 완료")

### 3.2 OISST Version 2: 선형 추세 먼저 제거 → 기후장 제거

```
O_detr(i,t) = O(i,t)      − (a_i + b_i·t)            ← 원본 SST의 선형 추세 제거
O'_v2(i,t)  = O_detr(i,t) − Ō_detr,1994₋₂₀₁₉(i,m)   ← 추세 제거 후 기후장 제거
```
이 방법은 원본 SST에서 온난화 추세를 먼저 제거한 다음,
추세가 없는 시계열에서 계절 사이클을 제거한다.

**v1과 v2의 차이**: 선형 추세와 계절 사이클이 상호 독립적이라면 결과는 같아야 한다.
하지만 실제로는 기후장 계산 기간(1994–2019)의 추세 기울기가 기후장 자체에 흡수되는 정도가
달라 미세한 차이가 생긴다.

In [ ]:
oisst_ssta_v2 = {}  # Version 2 결과 저장

for d in DOMAINS:
    da = oisst[d]  # 원본 OISST 월평균 SST

    # ── Step 1: 원본 SST에서 선형 추세 제거 ────────────────────────────
    da_detr = linear_detrend(da, dim="time")
    da_detr = da_detr.compute()

    # ── Step 2: 추세 제거된 시계열에서 1994–2019 월별 기후장 계산 ────────
    # 주의: v1과 달리 기후장을 계산하는 시계열 자체가 이미 detrend됨
    clim_period_detr = da_detr.sel(time=slice(CLIM_START, CLIM_END))
    clim_v2          = clim_period_detr.groupby("time.month").mean("time")

    # ── Step 3: 기후장 제거 ────────────────────────────────────────────
    ssta_v2 = da_detr.groupby("time.month") - clim_v2
    if "month" in ssta_v2.coords:
        ssta_v2 = ssta_v2.drop_vars("month")

    ssta_v2.attrs.update({
        "long_name"  : "OISST SSTA v2 (detrend → anomaly)",
        "units"      : "degC",
        "processing" : f"1) linear detrend per grid cell, "
                       f"2) monthly climatology removed ({CLIM_START}–{CLIM_END} of detrended)",
    })

    oisst_ssta_v2[d] = ssta_v2
    print(f"OISST v2 domain {d}: {dict(ssta_v2.sizes)}  "
          f"mean={float(ssta_v2.mean()):.4f}  std={float(ssta_v2.std()):.4f}")

print("\nOISST v2 계산 완료")

### 3.3 v1 vs v2 비교 시각화

임의 격자점의 시계열과 공간 평균 시계열을 비교해 두 버전의 차이를 확인한다.

In [ ]:
# ── 공간 평균 시계열 비교 (domain c) ─────────────────────────────────
d_check = "north_pacific"

v1_mean = oisst_ssta_v1[d_check].mean(dim=["lat", "lon"])
v2_mean = oisst_ssta_v2[d_check].mean(dim=["lat", "lon"])
diff    = v1_mean - v2_mean  # 두 버전의 차이

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

ax = axes[0]
v1_mean.plot(ax=ax, color="steelblue", lw=0.9, label="v1 (anomaly→detrend)")
v2_mean.plot(ax=ax, color="darkorange", lw=0.9, ls="--", label="v2 (detrend→anomaly)")
ax.axhline(0, color="black", lw=0.7)
ax.set_title(f"OISST SSTA domain average — domain {d_check}")
ax.set_ylabel("SSTA (°C)")
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

ax = axes[1]
diff.plot(ax=ax, color="purple", lw=0.9)
ax.axhline(0, color="black", lw=0.7)
ax.set_title("v1 - v2 diff")
ax.set_ylabel("diff (°C)")
ax.grid(alpha=0.3)

# 월별 바이어스 비교 (둘 다 0에 가까워야 정상)
ax = axes[2]
months = np.arange(1, 13)
bias_v1 = v1_mean.groupby("time.month").mean("time").values
bias_v2 = v2_mean.groupby("time.month").mean("time").values
width = 0.35
ax.bar(months - width/2, bias_v1, width, color="steelblue",  label="v1", alpha=0.8)
ax.bar(months + width/2, bias_v2, width, color="darkorange", label="v2", alpha=0.8)
ax.axhline(0, color="black", lw=0.7)
ax.set_xticks(months)
ax.set_xticklabels(["Jan","Feb","Mar","Apr","May","Jun",
                    "Jul","Aug","Sep","Oct","Nov","Dec"])
ax.set_title("Monthly bias (≈0이어야 정상)")
ax.set_ylabel("Monthly average SSTA (°C)")
ax.legend(fontsize=9)
ax.grid(alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig(FIG_DIR / f"oisst_v1_v2_compare_{d_check}.png", dpi=150)
plt.show()

print(f"v1 std: {float(v1_mean.std()):.4f} °C")
print(f"v2 std: {float(v2_mean.std()):.4f} °C")
print(f"v1-v2 RMSE: {float((diff**2).mean()**0.5):.6f} °C")

In [ ]:
# ── OISST SSTA 저장 ───────────────────────────────────────────────────
print("=== OISST SSTA 저장 ===")
for d in DOMAINS:
    # Version 1
    out_v1 = PROC_DIR / f"oisst_ssta_v1_{d}.nc"
    if out_v1.exists(): out_v1.unlink()
    oisst_ssta_v1[d].to_dataset(name="SSTA").to_netcdf(
        out_v1,
        encoding={"SSTA": {"zlib": True, "complevel": 4, "dtype": "float32"}},
    )
    print(f"저장: {out_v1.name}")

    # Version 2
    out_v2 = PROC_DIR / f"oisst_ssta_v2_{d}.nc"
    if out_v2.exists(): out_v2.unlink()
    oisst_ssta_v2[d].to_dataset(name="SSTA").to_netcdf(
        out_v2,
        encoding={"SSTA": {"zlib": True, "complevel": 4, "dtype": "float32"}},
    )
    print(f"저장: {out_v2.name}")

print("\nOISST SSTA 저장 완료")

## 4. 격자 맞추기 (CESM-HR → OISST grid)

### 배경: 왜 regrid가 필요한가?

- **CESM-HR**: POP 해양 모델의 **삼중극(tripolar) 곡선격자** (nlat×nlon의 2D TLAT/TLONG)
  - 격자 간격 ≈ 0.1°, 형태가 불규칙함
- **OISST**: **규칙 격자** (lat×lon 1D 배열, 0.25°×0.25°)

Analog selection에서 두 자료를 직접 비교하려면 동일한 격자로 맞춰야 한다.
CESM-HR SSTA를 OISST 격자로 변환한다.

### 방법: scipy.interpolate.LinearNDInterpolator

1. CESM-HR의 TLAT/TLONG를 1D 포인트 배열로 평탄화
2. `LinearNDInterpolator`로 삼각형 분할(Delaunay triangulation) **1회** 생성
3. 각 time step마다 SST값을 새로 넣어 OISST 격자점에서의 값 계산

> ✅ 삼각형 분할은 격자 구조(좌표)에만 의존하므로 1번만 생성하면 된다.

In [ ]:
from scipy.interpolate import LinearNDInterpolator
from scipy.spatial import Delaunay


def regrid_curvilinear_to_regular(
    da_src,
    src_lat_2d,
    src_lon_2d,
    tgt_lat_1d,
    tgt_lon_1d,
    time_dim="time",
):
    """
    곡선격자(curvilinear) DataArray를 규칙 격자(regular)로 선형 보간한다.

    Parameters
    ----------
    da_src     : xr.DataArray  — 소스 데이터, time_dim을 포함한 3D
    src_lat_2d : np.ndarray    — 소스 격자의 위도 (nlat, nlon)
    src_lon_2d : np.ndarray    — 소스 격자의 경도 (nlat, nlon)
    tgt_lat_1d : np.ndarray    — 목표 격자의 위도 1D 배열
    tgt_lon_1d : np.ndarray    — 목표 격자의 경도 1D 배열
    time_dim   : str           — 시간 차원 이름

    Returns
    -------
    xr.DataArray — 목표 격자로 보간된 배열 (time, lat, lon)
    """
    # ── time을 첫 번째 축으로 보장 ────────────────────────────────────
    # apply_ufunc 등 이전 연산에서 dim 순서가 바뀌었을 수 있으므로 명시적으로 정렬
    # transpose: time을 앞으로, 나머지 공간 차원은 뒤에
    spatial_dims = [d for d in da_src.dims if d != time_dim]
    da_src = da_src.transpose(time_dim, *spatial_dims)

    # ── numpy 추출 ────────────────────────────────────────────────────
    vals_3d = da_src.values  # (time, nlat, nlon) — 보장됨
    n_time  = vals_3d.shape[0]

    # 소스 격자 포인트 1D 평탄화
    src_lat_flat = src_lat_2d.ravel()   # (nlat*nlon,)
    src_lon_flat = src_lon_2d.ravel()   # (nlat*nlon,)

    # 목표 격자: meshgrid → (N_tgt, 2) 포인트 배열
    tgt_lon_2d, tgt_lat_2d = np.meshgrid(tgt_lon_1d, tgt_lat_1d)
    tgt_points = np.column_stack([tgt_lat_2d.ravel(), tgt_lon_2d.ravel()])
    n_lat, n_lon = len(tgt_lat_1d), len(tgt_lon_1d)

    # ── shape 일치 검증 ───────────────────────────────────────────────
    n_src_spatial = vals_3d[0].size   # nlat * nlon
    assert n_src_spatial == len(src_lat_flat), (
        f"소스 데이터 공간 크기({n_src_spatial})와 "
        f"src_lat_2d 크기({len(src_lat_flat)})가 다릅니다. "
        f"da_src.dims = {da_src.dims}"
    )

    # ── 해양 격자 마스크 (육지=NaN 제외) ─────────────────────────────
    valid_mask = np.isfinite(vals_3d[0].ravel())  # (nlat*nlon,)

    src_points_valid = np.column_stack([
        src_lat_flat[valid_mask],
        src_lon_flat[valid_mask],
    ])
    print(f"  유효 소스 포인트: {valid_mask.sum():,} / {len(valid_mask):,}")
    print(f"  목표 포인트    : {n_lat}×{n_lon} = {n_lat*n_lon:,}")

    # ── Delaunay 삼각분할 1회 생성 (이후 모든 time step에 재사용) ──────
    print(f"  Delaunay 삼각분할 생성 중...", end=" ", flush=True)
    tri = Delaunay(src_points_valid)
    print("완료")

    # ── time step별 보간 ──────────────────────────────────────────────
    result = np.full((n_time, n_lat, n_lon), np.nan, dtype=np.float32)

    for t in range(n_time):
        if t % 100 == 0:
            print(f"  보간 중: {t+1}/{n_time}", end="\r", flush=True)

        vals_valid = vals_3d[t].ravel()[valid_mask]
        interp     = LinearNDInterpolator(tri, vals_valid)
        result[t]  = interp(tgt_points).astype(np.float32).reshape(n_lat, n_lon)

    print(f"\n  보간 완료: {n_time}개 time step")

    # ── xr.DataArray 반환 ─────────────────────────────────────────────
    return xr.DataArray(
        result,
        dims=[time_dim, "lat", "lon"],
        coords={
            time_dim: da_src[time_dim].values,
            "lat"   : tgt_lat_1d,
            "lon"   : tgt_lon_1d,
        },
    )


print("regrid 함수 정의 완료")
print("(time을 첫 번째 축으로 강제 transpose + shape 검증 포함)")


### 4.2 도메인별 CESM-HR SSTA Regridding

> ⚠️ **계산 시간**: domain a (517×750 소스, 240×320 목표, 1200 time step)는 상당히 오래 걸린다.
> Delaunay triangulation이 가장 무거운 단계이며, 그 이후 보간은 비교적 빠르다.

In [ ]:
cesm_ssta_regridded = {}  # Regrid 결과 저장

for d in DOMAINS:
    print(f"\n=== Domain {d} regridding ===")

    # ── 소스: CESM-HR SSTA (curvilinear grid) ────────────────────────
    da_src = cesm_ssta[d]  # (time, nlat, nlon)

    # CESM-HR의 2D 위경도 좌표 추출
    # (TLAT, TLONG는 POP 삼중극 격자의 2D 배열)
    src_lat = da_src.coords["TLAT"].values   # (nlat, nlon)
    src_lon = da_src.coords["TLONG"].values  # (nlat, nlon)

    # ── 목표: 해당 도메인의 OISST 0.25° 규칙 격자 ────────────────────
    tgt_lat = oisst[d].lat.values   # 1D
    tgt_lon = oisst[d].lon.values   # 1D

    # ── Regrid 수행 ───────────────────────────────────────────────────
    da_rg = regrid_curvilinear_to_regular(
        da_src,
        src_lat, src_lon,
        tgt_lat, tgt_lon,
    )

    da_rg.attrs.update({
        "long_name" : "CESM-HR SSTA regridded to OISST 0.25° grid",
        "units"     : "degC",
        "method"    : "LinearNDInterpolator (Delaunay triangulation, scipy)",
        "source"    : f"cesm_hr_ssta_domain{d}.nc → OISST domain{d} grid",
    })

    cesm_ssta_regridded[d] = da_rg

    nan_pct = float(da_rg.isnull().mean()) * 100
    print(f"  결과 shape: {dict(da_rg.sizes)}")
    print(f"  NaN% = {nan_pct:.1f}%  (CESM 도메인 경계 외부 + 육지)")

print("\n전체 Regridding 완료")


In [ ]:
# ── Regrid 결과 시각화 검증 ───────────────────────────────────────────
# 원본 CESM-HR SSTA(curvilinear)와 regrid 결과(regular)를 같은 시점에 비교
# 두 그림이 공간 패턴상 비슷하면 regrid가 정상적으로 됐다는 의미

d_check = "north_pacific"   # domain c (가장 작아 시각화에 적합)
t_idx   = 600   # 600번째 time step (전체 1200개월 중 중간 시점)

src = cesm_ssta[d_check].isel(time=t_idx)             # 원본 curvilinear
rg  = cesm_ssta_regridded[d_check].isel(time=t_idx)   # regrid 결과

# 색상 범위: 두 자료의 99th percentile 기준으로 통일
vmax = float(max(
    abs(src.values[np.isfinite(src.values)]).max(),
    abs(rg.values[np.isfinite(rg.values)]).max(),
) * 0.8)  # 80%로 약간 줄여 색 대비 강조

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 원본: CESM-HR curvilinear (pcolormesh에 2D TLAT/TLONG 전달)
ax = axes[0]
im = ax.pcolormesh(
    src.coords["TLONG"].values,  # (nlat, nlon) — X축
    src.coords["TLAT"].values,   # (nlat, nlon) — Y축
    src.values,
    cmap="RdBu_r", vmin=-vmax, vmax=vmax, shading="auto",
)
plt.colorbar(im, ax=ax, label="SSTA (°C)", shrink=0.85)
ax.set_title(f"Original CESM-HR (curvilinear POP grid)\nt_idx = {t_idx}")
ax.set_xlabel("Longitude (°E)")
ax.set_ylabel("Latitude (°N)")

# Regrid 결과: OISST 0.25° 규칙 격자 (1D lat/lon 전달)
ax = axes[1]
im = ax.pcolormesh(
    rg.lon.values,   # 1D — X축
    rg.lat.values,   # 1D — Y축
    rg.values,
    cmap="RdBu_r", vmin=-vmax, vmax=vmax, shading="auto",
)
plt.colorbar(im, ax=ax, label="SSTA (°C)", shrink=0.85)
ax.set_title(f"Regrid Results (OISST 0.25° Regular Grid)
t_idx = {t_idx}")
ax.set_xlabel("Longitude (°E)")
ax.set_ylabel("Latitude (°N)")

plt.tight_layout()
plt.savefig(FIG_DIR / f"regrid_check_{d_check}_t{t_idx}.png", dpi=150)
plt.show()

# 공간 평균으로도 비교 (regrid 전후 값이 크게 달라지면 안 됨)
src_mean = float(np.nanmean(src.values))
rg_mean  = float(np.nanmean(rg.values))
print(f"Domain average - original: {src_mean:.4f} °C  /  After regrid: {rg_mean:.4f} °C")
print(f"diff: {abs(src_mean - rg_mean):.6f} °C")


In [ ]:
# ── Regrid된 CESM-HR SSTA 저장 ────────────────────────────────────────
print("=== Regrid된 CESM-HR SSTA 저장 ===")
for d in DOMAINS:
    out_path = PROC_DIR / f"cesm_hr_ssta_regridded_{d}.nc"
    if out_path.exists(): out_path.unlink()

    cesm_ssta_regridded[d].to_dataset(name="SSTA").to_netcdf(
        out_path,
        encoding={"SSTA": {"zlib": True, "complevel": 4, "dtype": "float32"}},
    )
    print(f"저장 완료: {out_path.name}  {dict(cesm_ssta_regridded[d].sizes)}")

print("\n=== 최종 저장 파일 목록 ===")
for fname in sorted(PROC_DIR.glob("*ssta*.nc")):
    size_mb = fname.stat().st_size / 1e6
    ds_tmp  = xr.open_dataset(fname)
    var     = list(ds_tmp.data_vars)[0]
    sizes   = dict(ds_tmp[var].sizes)
    ds_tmp.close()
    print(f"  {fname.name:<45} {size_mb:7.1f} MB  {sizes}")

## 5. 요약

### 생성된 파일

| 파일 | 설명 | 다음 단계 |
|------|------|----------|
| `cesm_hr_ssta_domain{a,b,c}.nc` | CESM-HR SSTA (detrend + climatology 제거, curvilinear grid) | 내부 분석용 |
| `oisst_ssta_v1_domain{a,b,c}.nc` | OISST SSTA v1 (anomaly→detrend) | Analog selection 초기장 후보 |
| `oisst_ssta_v2_domain{a,b,c}.nc` | OISST SSTA v2 (detrend→anomaly) | Analog selection 초기장 후보 |
| `cesm_hr_ssta_regridded_domain{a,b,c}.nc` | CESM-HR SSTA → OISST 0.25° grid | **Analog selection 라이브러리** |

### 다음 단계 (04_analog_selection.ipynb)

1. OISST SSTA (v1 또는 v2)를 **관측 초기장**으로 사용
2. CESM-HR SSTA (regridded)를 **analog 라이브러리**로 사용
3. 두 자료가 이제 동일한 격자(OISST 0.25°)에 있으므로 직접 거리(RMSE 등) 계산 가능
4. 가장 유사한 CESM-HR 상태를 찾아 앙상블 예측 생성